In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import geopandas as gpd
# print(os.listdir("/content/drive/MyDrive/255 Final Project/Data/Tile2Net/Berkeley/network/Berkeley-Network-03-04-2026_23_03/"))

In [3]:
# read the shapefile
berkeley_gdf = gpd.read_file('/content/drive/MyDrive/255 Final Project/Data/Tile2Net/Berkeley/network/Berkeley-Network-03-04-2026_23_03/Berkeley-Network-03-04-2026_23_03.shp')

# reproject to WGS84 for web maps like Leaflet
berkeley_gdf = berkeley_gdf.to_crs(epsg=4326)

# write to GeoJSON
# berkeley_gdf.to_file("/content/drive/MyDrive/255 Final Project/Data/Tile2Net/Berkeley/network/BerekleyNetwork.geojson", driver="GeoJSON")

In [4]:
# read the shapefile
oaklandnorth_gdf = gpd.read_file('/content/drive/MyDrive/255 Final Project/Data/Tile2Net/OaklandTop/network/OaklandTop-Network-07-04-2026_21_09/OaklandTop-Network-07-04-2026_21_09.shp')

# reproject to WGS84 for web maps like Leaflet
oaklandnorth_gdf = oaklandnorth_gdf.to_crs(epsg=4326)

# write to GeoJSON
# oaklandnorth_gdf.to_file("/content/drive/MyDrive/255 Final Project/Data/Tile2Net/OaklandTop/network/OaklandTopSidewalk.geojson", driver="GeoJSON")

In [5]:
# read the shapefile
oaklandsouth_gdf = gpd.read_file('/content/drive/MyDrive/255 Final Project/Data/Tile2Net/OaklandBottom/network/OaklandBottom-Network-07-04-2026_23_28/OaklandBottom-Network-07-04-2026_23_28.shp')

# reproject to WGS84 for web maps like Leaflet
oaklandsouth_gdf = oaklandsouth_gdf.to_crs(epsg=4326)

# write to GeoJSON
# oaklandsouth_gdf.to_file("/content/drive/MyDrive/255 Final Project/Data/Tile2Net/OaklandBottom/network/OaklandTopSidewalk.geojson", driver="GeoJSON")

In [6]:
combined_gdf = gpd.pd.concat([berkeley_gdf, oaklandnorth_gdf, oaklandsouth_gdf], ignore_index=True)

In [ ]:
import matplotlib.pyplot as plt
import folium

combined_gdf.plot(figsize=(8, 8))
plt.show()

# Center map on data
center = [combined_gdf.geometry.centroid.y.mean(), combined_gdf.geometry.centroid.x.mean()]

m = folium.Map(location=center, zoom_start=12)

# Add GeoJSON layer
folium.GeoJson(combined_gdf).add_to(m)

m

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Read files
points = gpd.read_file('/content/drive/MyDrive/255 Final Project/Data/stations_26.geojson')
lines = combined_gdf

# Make sure both layers use the same CRS
if points.crs != lines.crs:
    lines = lines.to_crs(points.crs)

# Important: buffer distance should be in a projected CRS with feet/meters,
# not plain lat/lon degrees
# Example: reproject both to a metric CRS first
points = points.to_crs(epsg=3857)
lines = lines.to_crs(epsg=3857)

# Buffer points by 500 meters
point_buffers = points.buffer(500)

# Combine all buffers into one shape
buffer_union = point_buffers.union_all()

# Keep only lines that intersect the buffered area
lines_near_points = lines[lines.intersects(buffer_union)]

# Plot
ax = lines_near_points.plot(figsize=(8, 8))
points.plot(ax=ax, color="red", markersize=20)
plt.show()